In [5]:
import pandas as pd
import numpy as np

file_path = r"c:\Users\kk\Desktop\Data Science\ml\Supply_chain\raw_data\DataCoSupplyChainDataset.csv"

df = pd.read_csv(file_path, encoding="latin1")

print("Shape:", df.shape)
df.head()

Shape: (180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [6]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

Shape: (180519, 53)

Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product Sta

In [7]:
df.columns = (
    df.columns
    .str.lower()
    .str.strip()
    .str.replace(" ", "_")
    .str.replace(r"[()]", "", regex=True)
)

In [8]:
print(df["late_delivery_risk"].value_counts())

late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64


In [9]:
# print(
#     df["late_delivery_risk"]
#     .value_counts(normalize=True)
#     .mul(100)
#     .round(2)
# )
df["late_delivery_risk"].value_counts(normalize=True).mul(100).round(2)

late_delivery_risk
1    54.83
0    45.17
Name: proportion, dtype: float64

In [11]:
missing=df.isnull().sum()
missing[missing>0].sort_values(ascending=False)

product_description    180519
order_zipcode          155679
customer_lname              8
customer_zipcode            3
dtype: int64

In [12]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [13]:
leakage_columns=["days_for shiping_real","delivery_status",
    "shipping_date_dateorders",
    "order_status"]
df=df.drop(columns=leakage_columns,errors='ignore')


In [14]:
drop_columns = [
    "customer_email",
    "customer_fname",
    "customer_lname",
    "customer_password",
    "customer_street",
    
    "customer_id",
    "order_customer_id",
    "order_id",
    "order_item_id",
    
    "product_description",
    "product_image"
]

df = df.drop(columns=drop_columns, errors="ignore")

print("Shape:", df.shape)

Shape: (180519, 39)


In [15]:
for i, col in enumerate(df.columns, start=1):
    print(i, col)

1 type
2 days_for_shipping_real
3 days_for_shipment_scheduled
4 benefit_per_order
5 sales_per_customer
6 late_delivery_risk
7 category_id
8 category_name
9 customer_city
10 customer_country
11 customer_segment
12 customer_state
13 customer_zipcode
14 department_id
15 department_name
16 latitude
17 longitude
18 market
19 order_city
20 order_country
21 order_date_dateorders
22 order_item_cardprod_id
23 order_item_discount
24 order_item_discount_rate
25 order_item_product_price
26 order_item_profit_ratio
27 order_item_quantity
28 sales
29 order_item_total
30 order_profit_per_order
31 order_region
32 order_state
33 order_zipcode
34 product_card_id
35 product_category_id
36 product_name
37 product_price
38 product_status
39 shipping_mode


In [16]:
remove_columns = [
    "days_for_shipping_real",      # leakage
    "benefit_per_order",            # derived after transaction
    "sales_per_customer",           # derived metric
    "order_item_profit_ratio",      # derived metric
    "order_item_total",             # derived from other fields
    "order_profit_per_order",       # derived metric

    "customer_city",                # too granular
    "customer_state",               # redundant with geography
    "customer_zipcode",              # too granular

    "order_city",                   # too granular
    "order_state",                  # too granular
    "order_zipcode",                # too granular

    "latitude",                     # unnecessary for first version
    "longitude",

    "order_item_cardprod_id",       # identifier
    "product_card_id",              # identifier
    "product_category_id",          # redundant with category_id

    "product_name",                 # high-cardinality
    "product_status",               # little value / potentially leakage
]

In [17]:
df = df.drop(columns=remove_columns, errors="ignore")

In [18]:
df = df.drop(columns=[
    "category_id",
    "department_id"
], errors="ignore")

In [28]:
df.groupby("shipping_mode")["late_delivery_risk"].mean() * 100

shipping_mode
First Class       95.322499
Same Day          45.743042
Second Class      76.632781
Standard Class    38.071683
Name: late_delivery_risk, dtype: float64

In [19]:
features = [
    "type",
    "days_for_shipment_scheduled",
    "category_name",
    "customer_segment",
    "department_name",
    "market",
    "order_country",
    "order_region",
    "order_item_discount",
    "order_item_discount_rate",
    "order_item_product_price",
    "order_item_quantity",
    "sales",
    "product_price",
    "shipping_mode"
]

In [20]:
target = "late_delivery_risk"

In [25]:
def late_rate_by(column):
    result=(df.groupby(column)["late_delivery_risk"]
            .agg(["count","mean"])
            .rename(columns={"mean":"late_rate"})
            .sort_values("late_rate",ascending=False))
    result["late_rate"]=(result["late_rate"]*100).round(2)
    return result

In [30]:
print("SHIPPING MODE")
display(late_rate_by("shipping_mode"))

print("SCHEDULED SHIPPING DAYS")
display(late_rate_by("days_for_shipment_scheduled"))

print("MARKET")
display(late_rate_by("market"))

print("ORDER REGION")
display(late_rate_by("order_region"))

print("CUSTOMER SEGMENT")
display(late_rate_by("customer_segment"))

print("CATEGORY")
display(late_rate_by("category_name"))

print("DEPARTMENT")
display(late_rate_by("department_name"))

SHIPPING MODE


,count,late_rate
shipping_mode,,
First Class,27814,95.32
Second Class,35216,76.63
Same Day,9737,45.74
Standard Class,107752,38.07


SCHEDULED SHIPPING DAYS


,count,late_rate
days_for_shipment_scheduled,,
1,27814,95.32
2,35216,76.63
0,9737,45.74
4,107752,38.07


MARKET


...

,count,late_rate
customer_segment,,
Home Office,32226,55.07
Consumer,93504,54.81
Corporate,54789,54.72


CATEGORY


,count,late_rate
category_name,,
Golf Bags & Carts,61,68.85
Lacrosse,343,60.06
Pet Supplies,492,58.94
Cameras,592,58.11
Strength Training,111,57.66
As Seen on TV!,68,57.35
Music,434,57.14
Accessories,1780,56.97
Fitness Accessories,309,56.96


DEPARTMENT


,count,late_rate
department_name,,
Pet Shop,492,58.94
Book Shop,405,56.54
Health and Beauty,362,55.80
Fitness,2479,55.55
Outdoors,9686,55.49
Technology,1465,55.02
Golf,33220,54.78
Fan Shop,66861,54.77
Apparel,48998,54.75


In [27]:
numerical_factors = [
    "order_item_quantity",
    "order_item_discount_rate",
    "order_item_product_price",
    "sales",
    "product_price"
]

df.groupby("late_delivery_risk")[numerical_factors].mean().round(2)

,order_item_quantity,order_item_discount_rate,order_item_product_price,sales,product_price
late_delivery_risk,,,,,
0,2.13,0.1,141.57,204.29,141.57
1,2.13,0.1,140.96,203.34,140.96


In [31]:
pd.crosstab(
    df["order_item_quantity"],
    df["late_delivery_risk"],
    normalize="index"
).round(3) * 100

late_delivery_risk,0,1
order_item_quantity,,
1,45.1,54.9
2,45.7,54.3
3,44.9,55.1
4,45.2,54.8
5,45.2,54.8


In [32]:
pd.crosstab(
    df["shipping_mode"],
    df["days_for_shipment_scheduled"]
)

days_for_shipment_scheduled,0,1,2,4
shipping_mode,,,,
First Class,0,27814,0,0
Same Day,9737,0,0,0
Second Class,0,0,35216,0
Standard Class,0,0,0,107752


In [33]:
df["order_date_dateorders"] = pd.to_datetime(
    df["order_date_dateorders"],
    errors="coerce"
)

df["order_month"] = df["order_date_dateorders"].dt.month

df.groupby("order_month")["late_delivery_risk"].mean() * 100

order_month
1     54.658212
2     54.552963
3     55.254727
4     54.279236
5     54.738358
6     54.825286
7     54.057279
8     55.794369
9     55.297308
10    54.596681
11    54.552000
12    55.335318
Name: late_delivery_risk, dtype: float64

In [34]:
df["order_dayofweek"] = df["order_date_dateorders"].dt.dayofweek

df.groupby("order_dayofweek")["late_delivery_risk"].mean() * 100

order_dayofweek
0    55.053130
1    54.300991
2    54.758276
3    55.005436
4    55.170685
5    54.395583
6    55.114469
Name: late_delivery_risk, dtype: float64

In [36]:
df.columns.tolist()

['type',
 'days_for_shipment_scheduled',
 'late_delivery_risk',
 'category_name',
 'customer_country',
 'customer_segment',
 'department_name',
 'market',
 'order_country',
 'order_date_dateorders',
 'order_item_discount',
 'order_item_discount_rate',
 'order_item_product_price',
 'order_item_quantity',
 'sales',
 'order_region',
 'product_price',
 'shipping_mode',
 'order_month',
 'order_dayofweek']

In [37]:
categorical_columns = [
    "type",
    "category_name",
    "customer_country",
    "customer_segment",
    "department_name",
    "market",
    "order_country",
    "order_region",
    "shipping_mode"
]

df[categorical_columns].nunique().sort_values()

customer_country      2
customer_segment      3
type                  4
shipping_mode         4
market                5
department_name      11
order_region         23
category_name        50
order_country       164
dtype: int64

In [38]:
pd.crosstab(
    df["customer_country"],
    df["order_country"],
    normalize="index"
).round(3)

order_country,Afganistán,Albania,Alemania,Angola,Arabia Saudí,Argelia,Argentina,Armenia,Australia,Austria,...,Ucrania,Uganda,Uruguay,Uzbekistán,Venezuela,Vietnam,Yemen,Yibuti,Zambia,Zimbabue
customer_country,,,,,,,,,,,,,,,,,,,,,
EE. UU.,0.001,0.0,0.052,0.002,0.004,0.003,0.011,0.0,0.047,0.007,...,0.005,0.001,0.001,0.001,0.006,0.004,0.0,0.0,0.002,0.001
Puerto Rico,0.001,0.0,0.054,0.002,0.005,0.002,0.010,0.0,0.046,0.008,...,0.006,0.000,0.001,0.001,0.005,0.004,0.0,0.0,0.001,0.001


In [39]:
categorical_columns = [
    "type",
    "category_name",
    "customer_country",
    "customer_segment",
    "department_name",
    "market",
    "order_country",
    "order_region",
    "shipping_mode"
]

In [40]:
numerical_columns = [
    "order_item_discount",
    "order_item_discount_rate",
    "order_item_product_price",
    "order_item_quantity",
    "sales",
    "product_price",
    "order_month",
    "order_dayofweek"
]

In [41]:
X = df[categorical_columns + numerical_columns]

y = df["late_delivery_risk"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (180519, 17)
y shape: (180519,)


In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (144415, 17)
X_test: (36104, 17)
y_train: (144415,)
y_test: (36104,)


In [43]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

In [44]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("X_train processed shape:", X_train_processed.shape)
print("X_test processed shape:", X_test_processed.shape)

X_train processed shape: (144415, 274)
X_test processed shape: (36104, 274)
